# 19. Внешняя проверка: HZMetro и SHMetro (Kaggle)

Метод заново обучается на открытых наборах из работы PVCGN (Ханчжоу 2019, Шанхай 2016) и сравнивается
с опубликованными результатами (`src/external.py`):

| запуск | постановка | с чем сравниваем |
|---|---|---|
| Ханчжоу, 15 мин | входы и выходы, 1–4 шага, MAE/RMSE/MAPE | PVCGN, HA, Graph WaveNet |
| Ханчжоу, 30 мин, только входы | 1–3 шага, WMAPE, тест 21–25.01.2019 | HW-DMD, HA |
| Шанхай, 15 мин | входы и выходы, 1–4 шага | PVCGN, HA, Graph WaveNet |

Данные скачиваются из репозитория авторов (github.com/ivechan/PVCGN). Датасет `vkr-transit-data` не нужен.
Accelerator — **GPU T4 x1** (для Chronos-2), Internet — On. Время ~1 ч. Результат — `results_csv.zip` в Output.

In [ ]:
!pip install -q chronos-forecasting==2.3.2

In [ ]:
import os, glob, shutil, subprocess, time
REPO = "/kaggle/working/vkr-transit"
if not os.path.exists(REPO):
    subprocess.run(["git", "clone", "--depth", "1", "https://github.com/Kinggl3/vkr-transit", REPO], check=True)
print(subprocess.run(["git", "-C", REPO, "log", "--oneline", "-1"], capture_output=True, text=True).stdout)
os.environ["VKR_RESULTS"] = R = "/kaggle/working/results"
os.environ["PYTHONWARNINGS"] = "ignore"
os.makedirs(R, exist_ok=True)

In [ ]:
DATA = "/kaggle/working/pvcgn"
if not os.path.exists(f"{DATA}/hangzhou"):
    subprocess.run(["git", "clone", "--depth", "1", "https://github.com/ivechan/PVCGN", "/kaggle/working/PVCGN"], check=True)
    os.makedirs(DATA, exist_ok=True)
    subprocess.run(["tar", "-xzf", "/kaggle/working/PVCGN/data/data.tar.gz", "-C", DATA, "--no-same-owner"], check=True)
print(sorted(glob.glob(f"{DATA}/*/*.pkl")))

## Ханчжоу

In [ ]:
!cd {REPO} && python src/external.py --data {DATA} --city hangzhou --step 15 --chronos

In [ ]:
!cd {REPO} && python src/external.py --data {DATA} --city hangzhou --step 30 --inflow-only --chronos

## Шанхай

In [ ]:
!cd {REPO} && python src/external.py --data {DATA} --city shanghai --step 15 --chronos

## Архив

In [ ]:
import zipfile
with zipfile.ZipFile("/kaggle/working/results_csv.zip", "w", zipfile.ZIP_DEFLATED) as z:
    for p in glob.glob(f"{R}/external_*.csv"):
        z.write(p, os.path.basename(p))
print(sorted(os.path.basename(p) for p in glob.glob(f"{R}/*")))